# Playground · Regresión Lineal

**Tópicos de Inteligencia de Negocios** · Análisis de Datos y Modelado Predictivo

> Este cuaderno es un **laboratorio interactivo**. Mueve los sliders, cambia parámetros y observa **en tiempo real** cómo cambia el modelo. La meta no es que memorices fórmulas — es que *sientas* cómo se comporta cada hiperparámetro.

¿Qué vas a poder hacer aquí?

1. Generar datos sintéticos con la relación que tú quieras (lineal, cuadrática, cúbica, senoidal) y controlar el ruido.
2. Subir tu propio CSV y ajustar regresión sobre tus datos.
3. Variar el **grado polinomial** del modelo y ver cuándo subajusta y cuándo sobreajusta.
4. Aplicar **regularización** (Ridge o Lasso) y observar cómo cambia la curva ajustada.
5. Comparar métricas (MSE, RMSE, MAE, R²) en **train** y **test**.

> 🔍 **Tip importante:** Junto a cada control verás un botón azul **`?`**. Haz clic en él para desplegar una explicación de qué hace ese parámetro. Vuelve a hacer clic para ocultarla. ¡Úsalo cada vez que tengas duda!


## Marco Teórico (lo esencial)

### El modelo
La regresión lineal asume una relación de la forma:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n$$

Si extendemos con **características polinomiales** ($x, x^2, x^3, \ldots$) podemos ajustar curvas no-lineales manteniendo el modelo lineal *en los parámetros*.

### Función de costo (MSE)
Buscamos los $\theta$ que minimicen el **Error Cuadrático Medio**:

$$J(\theta) = \frac{1}{m} \sum_{i=1}^{m} \left( \hat{y}^{(i)} - y^{(i)} \right)^2$$

### Regularización
Para evitar sobreajuste cuando el grado polinomial es alto, añadimos una penalización al costo:

| Tipo | Penalización | Efecto |
|------|--------------|--------|
| **Ridge (L2)** | $\alpha \sum \theta_j^2$ | Encoge todos los coeficientes hacia 0, sin anularlos |
| **Lasso (L1)** | $\alpha \sum \mid\theta_j\mid$ | Puede dejar coeficientes en *exactamente* 0 (selección de variables) |

`α` (alpha) controla qué tan fuerte es la penalización: $\alpha = 0$ → sin regularización; $\alpha$ muy grande → modelo subajustado.

### Métricas de evaluación

| Métrica | Fórmula | Interpretación |
|--------|---------|----------------|
| **MSE** | $\frac{1}{m}\sum(\hat{y}-y)^2$ | Error promedio al cuadrado (penaliza outliers) |
| **RMSE** | $\sqrt{\text{MSE}}$ | En las mismas unidades que $y$ |
| **MAE** | $\frac{1}{m}\sum\mid\hat{y}-y\mid$ | Error absoluto promedio (robusto a outliers) |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | % de varianza explicada (1 = perfecto, 0 = no aporta nada) |

> 💡 **Regla de oro:** si el desempeño en train es mucho mejor que en test → sobreajuste. Si ambos son malos → subajuste.


## 1. Configuración inicial

Ejecuta esta celda **una sola vez** al abrir el notebook.

In [ ]:
# ===== Auto-instalar paquetes que no vienen precargados en Pyodide (JupyterLite) =====
# Pyodide ya trae numpy, pandas, matplotlib y scikit-learn,
# pero ipywidgets hay que instalarlo en caliente la primera vez (~10 segundos).
try:
    import ipywidgets  # noqa: F401
except ImportError:
    import micropip  # disponible solo en Pyodide
    print('⏳ Instalando ipywidgets en el navegador (1 vez)...')
    await micropip.install('ipywidgets')
    print('✅ ipywidgets instalado')

# Librerías estándar de ciencia de datos
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Widgets interactivos
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Estilo de las gráficas
plt.rcParams['figure.dpi'] = 90
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Listo. Todas las librerías cargadas correctamente.')


## 2. Funciones auxiliares

Estas funciones se reusan en los dos playgrounds (datos sintéticos y CSV). Si te interesa cómo funciona cada una, ¡lee el código!

In [ ]:
# ==================================================================
# UI helper: agrega un botón "?" plegable junto a cualquier widget
# para mostrar/ocultar una explicación didáctica del parámetro.
# ==================================================================
def con_ayuda(control, explicacion):
    """Envuelve un widget con un botón ? que despliega ayuda contextual."""
    btn = widgets.Button(
        description='?', button_style='info', tooltip=explicacion,
        layout=widgets.Layout(width='30px', height='28px', margin='0 0 0 4px'),
    )
    panel = widgets.HTML(
        value=(f'<div style="background:#dbeafe; padding:8px 10px; '
               f'border-radius:4px; border-left:3px solid #2563eb; '
               f'margin:2px 0 8px 18px; font-size:12px; color:#1e3a8a;">'
               f'💡 {explicacion}</div>'),
        layout=widgets.Layout(display='none'),
    )
    def toggle(_):
        panel.layout.display = 'none' if panel.layout.display != 'none' else 'block'
    btn.on_click(toggle)
    return widgets.VBox([widgets.HBox([control, btn]), panel])


def generar_datos_sinteticos(n=100, ruido=10.0, relacion='Lineal',
                              pendiente=2.0, intercepto=0.0, seed=42):
    """Genera (X, y) sintéticos según la relación especificada."""
    rng = np.random.default_rng(seed)
    x = np.linspace(-10, 10, n)
    if relacion == 'Lineal':
        y_real = pendiente * x + intercepto
    elif relacion == 'Cuadrática':
        y_real = pendiente * 0.3 * x**2 + intercepto
    elif relacion == 'Cúbica':
        y_real = pendiente * 0.05 * x**3 + intercepto
    elif relacion == 'Senoidal':
        y_real = pendiente * 5 * np.sin(x) + intercepto
    else:
        raise ValueError(f'Relación desconocida: {relacion}')
    y = y_real + rng.normal(0, ruido, size=n)
    return x.reshape(-1, 1), y


def construir_modelo(grado, regularizacion, alpha):
    """Devuelve un Pipeline (PolynomialFeatures + modelo)."""
    if regularizacion == 'Ninguna':
        modelo = LinearRegression()
    elif regularizacion == 'Ridge (L2)':
        modelo = Ridge(alpha=max(alpha, 1e-8))
    elif regularizacion == 'Lasso (L1)':
        modelo = Lasso(alpha=max(alpha, 1e-8), max_iter=20000)
    else:
        raise ValueError(f'Regularización desconocida: {regularizacion}')
    return Pipeline([
        ('poly', PolynomialFeatures(degree=grado, include_bias=False)),
        ('modelo', modelo),
    ])


def calcular_metricas(y_train, y_pred_train, y_test, y_pred_test):
    """Devuelve un DataFrame con MSE/RMSE/MAE/R² para train y test."""
    def fila(y, yhat):
        mse = mean_squared_error(y, yhat)
        return [mse, np.sqrt(mse), mean_absolute_error(y, yhat), r2_score(y, yhat)]
    df = pd.DataFrame(
        [fila(y_train, y_pred_train), fila(y_test, y_pred_test)],
        index=['Train', 'Test'],
        columns=['MSE', 'RMSE', 'MAE', 'R²'],
    )
    return df.round(4)


def graficar_resultados(X, y, pipeline, X_train, X_test, y_train, y_test,
                        y_pred_train, y_pred_test, titulo=''):
    """Genera dos paneles: ajuste del modelo + residuales."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # --- Panel 1: scatter + curva ajustada ---
    x_grid = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)
    y_grid = pipeline.predict(x_grid)
    axes[0].scatter(X_train, y_train, alpha=0.7, s=35,
                    color='#2563eb', label='Train', edgecolor='white', linewidth=0.5)
    axes[0].scatter(X_test, y_test, alpha=0.85, s=45,
                    color='#dc2626', label='Test', marker='s',
                    edgecolor='white', linewidth=0.5)
    axes[0].plot(x_grid, y_grid, color='#16a34a', linewidth=2.5, label='Modelo ajustado')
    axes[0].set_title(f'Ajuste del modelo {titulo}', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('X'); axes[0].set_ylabel('y')
    axes[0].legend(loc='best', framealpha=0.9)

    # --- Panel 2: residuales ---
    res_train = y_train - y_pred_train
    res_test = y_test - y_pred_test
    axes[1].scatter(y_pred_train, res_train, alpha=0.7, s=35,
                    color='#2563eb', label='Train', edgecolor='white', linewidth=0.5)
    axes[1].scatter(y_pred_test, res_test, alpha=0.85, s=45,
                    color='#dc2626', label='Test', marker='s',
                    edgecolor='white', linewidth=0.5)
    axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.6, linewidth=1)
    axes[1].set_title('Residuales (y − ŷ)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicción ŷ'); axes[1].set_ylabel('Error')
    axes[1].legend(loc='best', framealpha=0.9)

    plt.tight_layout()
    plt.show()


## 3. Playground · Datos sintéticos 🧪

Aquí **tú generas los datos** moviendo los sliders. Es la forma más rápida de aislar conceptos: puedes responder preguntas como *"¿qué pasa con un grado polinomial muy alto cuando la relación verdadera es lineal?"* sin la incertidumbre de un dataset real.

**Lee primero:** los controles a la izquierda controlan los **datos**; los de la derecha controlan el **modelo**. Cuando termines de configurar, haz clic en el botón **🚀 Entrenar modelo** que aparece debajo para aplicar los cambios y ver las gráficas y métricas actualizadas.

> 🔍 **Tip:** junto a cada slider hay un **botón azul `?`**. Haz clic para desplegar una explicación de qué controla ese parámetro y por qué importa. Vuélvelo a clickear para ocultarla.


In [ ]:
# ----- Widgets para datos -----
w_n = widgets.IntSlider(value=100, min=20, max=500, step=10,
                        description='n puntos:', style={'description_width': '110px'})
w_ruido = widgets.FloatSlider(value=10.0, min=0.0, max=50.0, step=1.0,
                              description='Ruido (σ):', style={'description_width': '110px'})
w_relacion = widgets.Dropdown(options=['Lineal', 'Cuadrática', 'Cúbica', 'Senoidal'],
                              value='Lineal', description='Relación:',
                              style={'description_width': '110px'})
w_pend = widgets.FloatSlider(value=2.0, min=-3.0, max=3.0, step=0.1,
                             description='Pendiente:', style={'description_width': '110px'})
w_inter = widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.5,
                              description='Intercepto:', style={'description_width': '110px'})
w_seed = widgets.IntSlider(value=42, min=0, max=100, step=1,
                           description='Semilla:', style={'description_width': '110px'})

# ----- Widgets para modelo -----
w_grado = widgets.IntSlider(value=1, min=1, max=15, step=1,
                            description='Grado:', style={'description_width': '110px'})
w_reg = widgets.Dropdown(options=['Ninguna', 'Ridge (L2)', 'Lasso (L1)'],
                         value='Ninguna', description='Regulariz.:',
                         style={'description_width': '110px'})
w_alpha = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3, step=0.1,
                                 description='α (alpha):',
                                 style={'description_width': '110px'})
w_split = widgets.FloatSlider(value=0.2, min=0.1, max=0.5, step=0.05,
                              description='Test size:', style={'description_width': '110px'})

# ----- Explicaciones (clic en ? para mostrarlas en el notebook) -----
ayuda_n = ('Cantidad de datos a generar. Más puntos = mejor estimación, '
           'pero también más cómputo. Empieza con 100.')
ayuda_ruido = ('Desviación estándar (σ) del ruido gaussiano que se añade a y. '
               'σ=0 → datos perfectos sin ruido. σ alto → la señal se pierde entre el ruido.')
ayuda_relacion = ('Forma de la relación VERDADERA entre x e y. Cambiarla te '
                  'permite ver qué tan flexible necesita ser tu modelo. '
                  'Senoidal es la más complicada para una regresión lineal estándar.')
ayuda_pend = ('Magnitud/inclinación de la relación verdadera. Mayor pendiente '
              '= la curva crece más rápido. Negativa = decrece.')
ayuda_inter = ('Desplaza la relación verdadera hacia arriba o abajo (el b en y = mx + b).')
ayuda_seed = ('Semilla aleatoria. Misma semilla = mismos datos. Cámbiala para '
              'ver cómo se comporta tu modelo con OTRO dataset igual de difícil.')
ayuda_grado = ('Grado del polinomio del MODELO. Grado 1 = recta. Grados altos = '
               'curvas muy flexibles. Cuidado: grado muy alto sin regularización tiende a sobreajustar.')
ayuda_reg = ('Penalización aplicada a coeficientes grandes para evitar sobreajuste. '
             '• Ridge (L2): los encoge suavemente. • Lasso (L1): puede anularlos exactamente.')
ayuda_alpha = ('Fuerza de la regularización. α≈0 → casi sin penalización (puede sobreajustar). '
               'α grande → penaliza mucho (puede subajustar). Mueve el slider y observa los coeficientes.')
ayuda_split = ('Proporción de datos reservada para test (no se usa en el ajuste). '
               '0.2 significa 80% train / 20% test. Es lo que mide la generalización REAL.')

# ----- Composición visual -----
panel_datos = widgets.VBox([
    widgets.HTML('<b>📊 Datos</b>  <span style="color:#64748b;font-size:11px;">'
                 '(clic en <b>?</b> para ver qué hace cada control)</span>'),
    con_ayuda(w_n, ayuda_n),
    con_ayuda(w_ruido, ayuda_ruido),
    con_ayuda(w_relacion, ayuda_relacion),
    con_ayuda(w_pend, ayuda_pend),
    con_ayuda(w_inter, ayuda_inter),
    con_ayuda(w_seed, ayuda_seed),
])
panel_modelo = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>  <span style="color:#64748b;font-size:11px;">'
                 '(clic en <b>?</b> para ver qué hace cada control)</span>'),
    con_ayuda(w_grado, ayuda_grado),
    con_ayuda(w_reg, ayuda_reg),
    con_ayuda(w_alpha, ayuda_alpha),
    con_ayuda(w_split, ayuda_split),
])
controles = widgets.HBox([panel_datos, panel_modelo])

salida = widgets.Output()

def actualizar_sintetico(*_):
    with salida:
        clear_output(wait=True)
        # Datos
        X, y = generar_datos_sinteticos(
            n=w_n.value, ruido=w_ruido.value, relacion=w_relacion.value,
            pendiente=w_pend.value, intercepto=w_inter.value, seed=w_seed.value,
        )
        # Train/test
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=w_split.value, random_state=42)
        # Modelo
        pipe = construir_modelo(w_grado.value, w_reg.value, w_alpha.value)
        try:
            pipe.fit(X_tr, y_tr)
        except Exception as e:
            print(f'⚠️ Error al ajustar el modelo: {e}')
            return
        y_pred_tr = pipe.predict(X_tr)
        y_pred_te = pipe.predict(X_te)
        # Visualización
        graficar_resultados(X, y, pipe, X_tr, X_te, y_tr, y_te,
                            y_pred_tr, y_pred_te,
                            titulo=f'(grado={w_grado.value}, {w_reg.value})')
        # Métricas
        df = calcular_metricas(y_tr, y_pred_tr, y_te, y_pred_te)
        print('\n📈 Métricas')
        display(df.style.format('{:.4f}').background_gradient(
            cmap='RdYlGn_r', subset=['MSE', 'RMSE', 'MAE']
        ).background_gradient(cmap='RdYlGn', subset=['R²']))
        # Diagnóstico rápido
        r2_train = df.loc['Train', 'R²']; r2_test = df.loc['Test', 'R²']
        gap = r2_train - r2_test
        if gap > 0.15 and r2_train > 0.5:
            print('\n🔴 Diagnóstico: Posible SOBREAJUSTE — el modelo memoriza train pero falla en test.')
        elif r2_train < 0.3 and r2_test < 0.3:
            print('\n🟡 Diagnóstico: Posible SUBAJUSTE — el modelo es demasiado simple para los datos.')
        else:
            print('\n🟢 Diagnóstico: El modelo parece estar generalizando razonablemente bien.')

# ----- Botón "Entrenar modelo" e indicador de estado -----
btn_entrenar = widgets.Button(
    description='🚀 Entrenar modelo',
    button_style='primary',
    tooltip='Aplica la configuración actual y entrena el modelo',
    layout=widgets.Layout(width='220px', height='40px', margin='10px 0 6px 0'),
)
estado = widgets.HTML(
    value='<span style="color:#64748b;font-style:italic;">'
          'Configura los parámetros y haz clic en <b>Entrenar modelo</b>.</span>'
)

def _marcar_stale(*_):
    estado.value = ('<span style="color:#ea580c;">'
                    '🔄 <b>Cambios sin aplicar.</b> Haz clic en '
                    '<b>Entrenar modelo</b> para verlos reflejados.</span>')

def _click_entrenar(_):
    btn_entrenar.disabled = True
    btn_entrenar.description = '⏳ Entrenando...'
    estado.value = ('<span style="color:#2563eb;">'
                    '⏳ Entrenando con la configuración actual...</span>')
    try:
        actualizar_sintetico()
        estado.value = ('<span style="color:#16a34a;">'
                        '✅ <b>Modelo entrenado.</b> Revisa las gráficas y '
                        'métricas abajo. Cambia parámetros y vuelve a entrenar.</span>')
    except Exception as e:
        estado.value = f'<span style="color:#dc2626;">❌ Error: {e}</span>'
    finally:
        btn_entrenar.disabled = False
        btn_entrenar.description = '🚀 Entrenar modelo'

btn_entrenar.on_click(_click_entrenar)

# Cualquier cambio en los widgets marca el estado como "pendiente"
for w in [w_n, w_ruido, w_relacion, w_pend, w_inter, w_seed,
          w_grado, w_reg, w_alpha, w_split]:
    w.observe(_marcar_stale, names='value')

display(controles, btn_entrenar, estado, salida)
# Entrenamiento inicial para que veas algo al abrir el notebook
actualizar_sintetico()
estado.value = ('<span style="color:#16a34a;">'
                '✅ <b>Modelo entrenado con la configuración inicial.</b> '
                'Modifica los sliders y dale a <b>Entrenar modelo</b>.</span>')


## 4. Playground · Sube tu CSV 📂

Ahora con **datos reales**. Sube cualquier CSV con al menos dos columnas numéricas y elige cuál quieres usar como X y cuál como Y.

**Tips para tu CSV:**
- La primera fila debe ser el encabezado (nombres de columnas).
- Solo se mostrarán columnas **numéricas** en los dropdowns (X e Y).
- Si tu archivo es grande, considera muestrearlo antes de subirlo (la regresión polinomial puede tardar).

> 🔍 **Tip:** los botones `?` también están aquí — úsalos si tienes duda sobre qué hace cada control.


In [ ]:
# Estado compartido
estado_csv = {'df': None, 'X': None, 'y': None, 'col_x': None, 'col_y': None}

# Widgets de carga
w_upload = widgets.FileUpload(accept='.csv', multiple=False,
                              description='📁 Subir CSV')
w_col_x = widgets.Dropdown(options=[], description='Columna X:',
                           style={'description_width': '110px'})
w_col_y = widgets.Dropdown(options=[], description='Columna Y:',
                           style={'description_width': '110px'})

# Widgets de modelo (separados de los del playground 1)
w_grado2 = widgets.IntSlider(value=1, min=1, max=15, step=1,
                             description='Grado:', style={'description_width': '110px'})
w_reg2 = widgets.Dropdown(options=['Ninguna', 'Ridge (L2)', 'Lasso (L1)'],
                          value='Ninguna', description='Regulariz.:',
                          style={'description_width': '110px'})
w_alpha2 = widgets.FloatLogSlider(value=1.0, base=10, min=-3, max=3, step=0.1,
                                  description='α (alpha):',
                                  style={'description_width': '110px'})
w_split2 = widgets.FloatSlider(value=0.2, min=0.1, max=0.5, step=0.05,
                               description='Test size:', style={'description_width': '110px'})

salida_csv = widgets.Output()
salida_info = widgets.Output()

def on_upload(change):
    with salida_info:
        clear_output(wait=True)
        if not w_upload.value:
            return
        try:
            archivo = w_upload.value[0] if isinstance(w_upload.value, tuple) else next(iter(w_upload.value.values()))
            contenido = archivo['content'] if isinstance(archivo, dict) else archivo['content']
            nombre = archivo.get('name') if isinstance(archivo, dict) else archivo['metadata']['name']
        except Exception:
            archivo = list(w_upload.value.values())[0]
            contenido = archivo['content']
            nombre = archivo.get('metadata', {}).get('name', 'archivo.csv')
        try:
            df = pd.read_csv(io.BytesIO(bytes(contenido)))
        except Exception as e:
            print(f'⚠️ No pude leer el CSV: {e}')
            return
        estado_csv['df'] = df
        cols_num = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(cols_num) < 2:
            print(f'⚠️ Tu CSV tiene menos de 2 columnas numéricas. Encontradas: {cols_num}')
            return
        w_col_x.options = cols_num
        w_col_y.options = cols_num
        w_col_x.value = cols_num[0]
        w_col_y.value = cols_num[1] if len(cols_num) > 1 else cols_num[0]
        print(f'✅ Cargado: {nombre} — {df.shape[0]} filas × {df.shape[1]} columnas')
        print(f'   Columnas numéricas detectadas: {cols_num}')
        print('\n📋 Vista previa (primeras 5 filas):')
        display(df.head())

w_upload.observe(on_upload, names='value')

def actualizar_csv(*_):
    with salida_csv:
        clear_output(wait=True)
        df = estado_csv['df']
        if df is None:
            print('⬆️ Sube un CSV primero.')
            return
        col_x, col_y = w_col_x.value, w_col_y.value
        if not col_x or not col_y:
            print('Selecciona columnas X e Y.')
            return
        if col_x == col_y:
            print('⚠️ X y Y deben ser columnas distintas.')
            return
        sub = df[[col_x, col_y]].dropna()
        if len(sub) < 10:
            print(f'⚠️ Muy pocos datos ({len(sub)} filas) — al menos 10.')
            return
        X = sub[[col_x]].values
        y = sub[col_y].values
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=w_split2.value, random_state=42)
        pipe = construir_modelo(w_grado2.value, w_reg2.value, w_alpha2.value)
        try:
            pipe.fit(X_tr, y_tr)
        except Exception as e:
            print(f'⚠️ Error al ajustar: {e}')
            return
        y_pred_tr = pipe.predict(X_tr)
        y_pred_te = pipe.predict(X_te)
        graficar_resultados(X, y, pipe, X_tr, X_te, y_tr, y_te,
                            y_pred_tr, y_pred_te,
                            titulo=f'({col_y} vs {col_x})')
        df_m = calcular_metricas(y_tr, y_pred_tr, y_te, y_pred_te)
        print(f'\n📈 Métricas — {col_y} en función de {col_x}')
        display(df_m.style.format('{:.4f}').background_gradient(
            cmap='RdYlGn_r', subset=['MSE', 'RMSE', 'MAE']
        ).background_gradient(cmap='RdYlGn', subset=['R²']))

# ----- Botón "Entrenar modelo" e indicador de estado (CSV) -----
btn_entrenar2 = widgets.Button(
    description='🚀 Entrenar modelo',
    button_style='primary',
    tooltip='Aplica la configuración actual y entrena con tu CSV',
    layout=widgets.Layout(width='220px', height='40px', margin='10px 0 6px 0'),
)
estado2 = widgets.HTML(
    value='<span style="color:#64748b;font-style:italic;">'
          'Sube un CSV, elige columnas y haz clic en <b>Entrenar modelo</b>.</span>'
)

def _marcar_stale2(*_):
    if estado_csv['df'] is None:
        return
    estado2.value = ('<span style="color:#ea580c;">'
                     '🔄 <b>Cambios sin aplicar.</b> Haz clic en '
                     '<b>Entrenar modelo</b> para verlos reflejados.</span>')

def _click_entrenar2(_):
    if estado_csv['df'] is None:
        estado2.value = ('<span style="color:#dc2626;">'
                         '⚠️ Primero sube un CSV.</span>')
        return
    btn_entrenar2.disabled = True
    btn_entrenar2.description = '⏳ Entrenando...'
    estado2.value = ('<span style="color:#2563eb;">'
                     '⏳ Entrenando con la configuración actual...</span>')
    try:
        actualizar_csv()
        estado2.value = ('<span style="color:#16a34a;">'
                         '✅ <b>Modelo entrenado.</b> Cambia parámetros y vuelve a entrenar.</span>')
    except Exception as e:
        estado2.value = f'<span style="color:#dc2626;">❌ Error: {e}</span>'
    finally:
        btn_entrenar2.disabled = False
        btn_entrenar2.description = '🚀 Entrenar modelo'

btn_entrenar2.on_click(_click_entrenar2)

for w in [w_col_x, w_col_y, w_grado2, w_reg2, w_alpha2, w_split2]:
    w.observe(_marcar_stale2, names='value')

# ----- Explicaciones del Playground 2 -----
ay_upload = ('Sube un archivo .csv con encabezados en la primera fila. '
             'Solo las columnas numéricas estarán disponibles para X e Y.')
ay_colx = ('Variable INDEPENDIENTE (predictor). Es la entrada del modelo, '
           'el valor que conoces y usas para hacer la predicción.')
ay_coly = ('Variable DEPENDIENTE (objetivo). Es lo que quieres predecir, '
           'el resultado del modelo.')

panel_carga = widgets.VBox([
    widgets.HTML('<b>📁 Datos (CSV)</b>  <span style="color:#64748b;font-size:11px;">'
                 '(clic en <b>?</b> para ver qué hace cada control)</span>'),
    con_ayuda(w_upload, ay_upload),
    con_ayuda(w_col_x, ay_colx),
    con_ayuda(w_col_y, ay_coly),
])
panel_modelo2 = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>  <span style="color:#64748b;font-size:11px;">'
                 '(clic en <b>?</b> para ver qué hace cada control)</span>'),
    con_ayuda(w_grado2, ayuda_grado),
    con_ayuda(w_reg2, ayuda_reg),
    con_ayuda(w_alpha2, ayuda_alpha),
    con_ayuda(w_split2, ayuda_split),
])
controles_csv = widgets.HBox([panel_carga, panel_modelo2])
display(controles_csv, btn_entrenar2, estado2, salida_info, salida_csv)


## 5. Ejercicios guiados 📝

Realiza los siguientes experimentos y **anota tus respuestas** (en una celda de markdown nueva o en un documento aparte que entregarás como evidencia).

### Ejercicio 1 — Subajuste vs Sobreajuste
1. En el playground sintético, elige **Relación = Cúbica**, n = 100, ruido = 8.
2. Empieza con grado polinomial = 1. Observa el modelo y las métricas.
3. Ahora prueba grado = 3, luego = 7, luego = 14.
4. **Pregunta:** ¿En qué grado el modelo *generaliza* mejor (R² de test más alto)? ¿Por qué grados muy altos empeoran el test?

### Ejercicio 2 — Efecto del ruido
1. Relación = Lineal, grado = 1.
2. Mueve el ruido de 0 a 50.
3. **Pregunta:** ¿Cómo cambia el R²? ¿Qué métrica (MSE, MAE) se ve más afectada por los outliers que aparecen con ruido alto?

### Ejercicio 3 — Regularización al rescate
1. Relación = Senoidal, n = 50, ruido = 5.
2. Pon **grado = 12** y **regularización = Ninguna**. Observa el desastre.
3. Cambia a **Ridge (L2)** y mueve `alpha` de 0.001 a 100.
4. **Pregunta:** ¿En qué valor de alpha se "domestica" la curva? ¿Qué pasa si alpha es muy grande?

### Ejercicio 4 — Lasso vs Ridge
Con la misma config del ejercicio 3, compara **Ridge** vs **Lasso** con `alpha = 1`.

**Pregunta:** Inspecciona los coeficientes con la celda de abajo. ¿Cuál tipo de regularización deja más coeficientes en cero?

### Ejercicio 5 — Tu propio dataset
1. Sube un CSV de tu interés (ventas mensuales, precios, datos de algún experimento).
2. Encuentra una pareja X, Y donde la regresión lineal funcione razonablemente bien (R² > 0.5).
3. **Pregunta:** ¿La relación es realmente lineal o necesitaste subir el grado? Justifica con la gráfica de residuales.


In [ ]:
# 🔬 Inspector de coeficientes (úsalo para el Ejercicio 4)
# Después de ajustar un modelo en el Playground 1, ejecuta esta celda
# para ver los coeficientes que aprendió.

X_, y_ = generar_datos_sinteticos(
    n=w_n.value, ruido=w_ruido.value, relacion=w_relacion.value,
    pendiente=w_pend.value, intercepto=w_inter.value, seed=w_seed.value,
)
X_tr_, X_te_, y_tr_, y_te_ = train_test_split(X_, y_, test_size=w_split.value, random_state=42)
pipe_ = construir_modelo(w_grado.value, w_reg.value, w_alpha.value)
pipe_.fit(X_tr_, y_tr_)

modelo = pipe_.named_steps['modelo']
poly = pipe_.named_steps['poly']
nombres = poly.get_feature_names_out(['x'])
coefs = pd.DataFrame({
    'Característica': nombres,
    'Coeficiente': modelo.coef_,
    '|Coef|': np.abs(modelo.coef_),
}).sort_values('|Coef|', ascending=False).reset_index(drop=True)

print(f'Intercepto (θ₀): {modelo.intercept_:.4f}')
print(f'Regularización: {w_reg.value} (α = {w_alpha.value:.4f})')
print(f'\nNº coeficientes exactamente en 0: {(coefs["Coeficiente"] == 0).sum()} de {len(coefs)}')
display(coefs.style.format({'Coeficiente': '{:.4f}', '|Coef|': '{:.4f}'}))


## 6. Resumen y siguientes pasos

### Lo que aprendiste jugando aquí

- La **regresión lineal** se puede extender a relaciones no-lineales con características polinomiales.
- A mayor **grado polinomial**, más capacidad — pero también más riesgo de **sobreajuste**.
- La **regularización** (Ridge/Lasso) penaliza coeficientes grandes y mejora la generalización.
- **R² en test** es la métrica más honesta para juzgar un modelo: puede mentirte en train, no en test.
- Los **residuales** deberían verse como ruido aleatorio alrededor de cero — si ves un patrón, te falta capacidad o transformaciones.

### Próximos modelos a explorar
Cuando termines este, abre los otros notebooks del playground:

- 🎯 **Regresión Logística** — clasificación binaria con sigmoide
- 🌳 **Árboles de Decisión** — preguntas encadenadas y particiones
- 🏘️ **KNN** — votación por vecindad
- 🎨 **K-Means** — descubrir grupos sin etiquetas

---

> *Tópicos de Inteligencia de Negocios · Playground de Machine Learning*
